# Ноутбук графиков по результатам MD


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import matplotlib.colors as mcolors
import seaborn as sns
from matplotlib.legend_handler import HandlerTuple
import matplotlib.lines as mlines
from brokenaxes import brokenaxes

In [ ]:
system_wt = "7NEI"
system_wt_mhet = "7NEI_MHET"
system_nc_mhet = "NC_7NEI_MHET"
system_nc = "NC_7NEI"

home_path = "8sys/" # путь до рабочей папки со всеми системами
plt.rcParams['figure.dpi'] = 200

#### RMSD

Использованные ниже функции применимы для всех графиков с RMSD(t) и распределением RMSD

In [ ]:
def read_rmsd_tables (path): #чтение таблиц с данными по RMSD 
    tables = []
    for i in range(4):
        tables.append(pd.read_csv(path+f"rep{i+1}_rmsd.tsv", sep = "\t"))
    return tables


def extract_cols (tables, group, subgroup =""): #выделение из таблиц колонок с нужными данными
    res_cols = tables[0][group+subgroup+"_MA"]
    repeats_num = len(tables)
    repeats = [f"Rep{i+1}" for i in range(repeats_num)]

    for table in tables[1::]:
        res_cols = pd.concat([res_cols, table[group+subgroup+"_MA"]], axis = 1)

    res_cols.columns = repeats
    return res_cols

def plotit_new (df_wt, df_nc,title = "", ylim = 10): #визуализация кривых RMSD(t)
    nc_cmap = sns.light_palette("#88BAF2", n_colors=4, as_cmap=True)
    nc_palette = nc_cmap(np.linspace(0.3, 1, 4))
    wt_cmap = sns.light_palette("#CD1C18", n_colors=4, as_cmap = True)
    wt_palette = wt_cmap(np.linspace(0.3, 1, 4))

    df_wt.reset_index(names="Frame", inplace=True)
    df_nc.reset_index(names="Frame", inplace=True)

    plt.figure(figsize=(7,5))
    pal = 0
    for i in df_wt.columns[1::]:
        plt.plot(df_wt["Frame"]*0.02, df_wt[i], color=wt_palette[pal])
        pal+=1

    pal = 0
    for i in df_nc.columns[1::]:
        plt.plot(df_nc["Frame"]*0.02, df_nc[i], color=nc_palette[pal])
        pal+=1


    blue_handles = tuple(
        mlines.Line2D([], [], color=c, lw=2) for c in nc_palette
    )

    red_handles = tuple(
        mlines.Line2D([], [], color=c, lw=2) for c in wt_palette
    )

    plt.legend(
        [red_handles, blue_handles],
        [ 'WT PHL7','trPHL7'],
        handler_map={tuple: HandlerTuple(ndivide=None)}
    )

    plt.title(title,fontsize=15)
    plt.xlabel("Время, нс",fontsize=15)
    plt.ylabel("RMSD, Å",fontsize=15)
    plt.xticks(fontsize=13)
    plt.yticks(fontsize=13)
    plt.ylim((0, ylim))
    plt.show()
    return None


def plot_violin (df_wt, df_nc, title = "", ylim = 10): #визуализация распределения RMSD

    violin_data = pd.DataFrame({
    "trPHL7": df_nc.drop("Frame", axis = 1).melt().dropna()["value"],
    "WT PHL7": df_wt.drop("Frame", axis = 1).melt().dropna()["value"]
    })
    violin_data = violin_data.melt(var_name="system", value_name="RMSD")
    violin_data["sp"] = ""

    plt.figure(figsize=(3, 5))
    sns.violinplot(data=violin_data,x ="sp", y = "RMSD", hue="system", gap = 0.04,split = True, inner=None, palette=["#88BAF2","#CD1C18"])
    plt.ylim((0,ylim))
    plt.ylabel("RMSD, Å", fontsize=15)
    plt.yticks(fontsize=13)

    ax = plt.gca()
    ax.get_xaxis().set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.get_legend().remove()
    plt.show()
    
    return None


In [ ]:
wt_300 = read_rmsd_tables(home_path+system_wt+"/343K/tables/") #чтение данных WT
nc_300 = read_rmsd_tables(home_path+system_nc+"/343K/tables/") #чтение данных NC

##### Backbone

In [ ]:
#Пример вызова для визуализации RMSD остова
bb_wt = extract_cols(wt_300, "Backbone")
bb_nc = extract_cols(nc_300, "Backbone")

plotit_new(bb_wt, bb_nc, title = "+МГЭТ 343K", ylim = 4.2)

In [ ]:
plot_violin(bb_wt, bb_nc)

RMSD триады (дополнительно нужны были разрывные оси)

In [ ]:
df_wt = extract_cols(wt_300, "Triad SC")
df_nc = extract_cols(nc_300, "Triad SC")

nc_cmap = sns.light_palette("#88BAF2", n_colors=4, as_cmap=True)
nc_palette = nc_cmap(np.linspace(0.3, 1, 4))
wt_cmap = sns.light_palette("#CD1C18", n_colors=4, as_cmap = True)
wt_palette = wt_cmap(np.linspace(0.3, 1, 4))

df_wt.reset_index(names="Frame", inplace=True)
df_nc.reset_index(names="Frame", inplace=True)

plt.figure(figsize=(7,5))

bax = brokenaxes(
ylims=((0, 4.2), (7.5, 10)),
hspace=0.05   # расстояние между кусками\
)
pal = 0
for i in df_wt.columns[1:]:
    bax.plot(df_wt["Frame"]*0.02, df_wt[i], color=wt_palette[pal])
    pal += 1

# --- NC ---
pal = 0
for i in df_nc.columns[1:]:
    bax.plot(df_nc["Frame"]*0.02, df_nc[i], color=nc_palette[pal])
    pal += 1

# --- легенда ---
blue_handles = tuple(
    mlines.Line2D([], [], color=c, lw=2) for c in nc_palette
)

red_handles = tuple(
    mlines.Line2D([], [], color=c, lw=2) for c in wt_palette
)

bax.legend(
    [red_handles, blue_handles],
    ['WT PHL7', 'trPHL7'],
    handler_map={tuple: HandlerTuple(ndivide=None)}
)

# --- подписи ---
bax.set_xlabel("Время, нс", fontsize=15,labelpad=20)
bax.set_ylabel("RMSD, Å", fontsize=15)

for ax in bax.axs:
    ax.spines['right'].set_visible(True)
    ax.tick_params(axis='x', labelsize=15)
    ax.tick_params(axis='y', labelsize=15)

bax.axs[0].spines['top'].set_visible(True)

plt.title("343K", fontsize=15)
plt.show()

In [ ]:
plot_violin(df_wt, df_nc, ylim = 10)

#### Расстояния в триаде

In [ ]:
def plot_replicas (df, palette, ax):
    pal = 0
    for i in df.filter(regex="_MA").columns:
        ax.plot(df["Frame"]*0.02, df[i], color=palette[pal])
        pal+=1
    return None

def dist_new (ser_wt, ser_nc, asp_wt, asp_nc, title = "", ylim = 5):

    nc_cmap = sns.light_palette("#88BAF2", n_colors=4, as_cmap=True)
    nc_palette = nc_cmap(np.linspace(0.3, 1, 4))
    wt_cmap = sns.light_palette("#CD1C18", n_colors=4, as_cmap = True)
    wt_palette = wt_cmap(np.linspace(0.3, 1, 4))
    
    if "Frame" not in ser_wt.columns:
        ser_wt.reset_index(names="Frame", inplace=True)
        ser_nc.reset_index(names="Frame", inplace=True)
        asp_wt.reset_index(names="Frame", inplace=True)
        asp_nc.reset_index(names="Frame", inplace=True)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6),sharey=True)
    plot_replicas(ser_wt, wt_palette, ax1)
    plot_replicas(ser_nc, nc_palette, ax1)
    plot_replicas(asp_wt, wt_palette, ax2)
    plot_replicas(asp_nc, nc_palette, ax2)


    ax1.axhline(y = 3.5, color = "grey", linestyle = "--", linewidth = 0.6)
    ax2.axhline(y = 3.5, color = "grey", linestyle = "--", linewidth = 0.6)

    ax1.set_title("131Ser Oγ – Nε\u2082 209His")
    ax2.set_title("209His Nδ\u2081 – Oδ\u2081 177Asp")
    ax1.tick_params(labelsize=13)
    ax2.tick_params(labelsize=13)

    blue_handles = tuple(
        mlines.Line2D([], [], color=c, lw=2) for c in nc_palette
    )

    red_handles = tuple(
        mlines.Line2D([], [], color=c, lw=2) for c in wt_palette
    )

    fig.legend(
        [red_handles, blue_handles],
        [ 'WT PHL7','trPHL7'],
        handler_map={tuple: HandlerTuple(ndivide=None)},
        bbox_to_anchor=(0.9, 0.98)
    )
   
    plt.subplots_adjust(hspace = 0.3)
    fig.suptitle(title,fontsize=15)
    plt.xlabel("Время, нс",fontsize=15)
    fig.supylabel("Расстояние, Å", fontsize=15)
    plt.ylim((2.5, 5.5))
    plt.show()
    return None

def plot_violin_2panels(df1_wt, df1_nc, df2_wt, df2_nc, 
                       title1="", title2="", ylim=10): #построение таких же двух графиков распределения расстояния

    def prepare_data(df_wt, df_nc):
        violin_data = pd.DataFrame({
            "trPHL7": df_nc.filter(regex = "_MA").melt().dropna()["value"],
            "WT PHL7": df_wt.filter(regex = "_MA").melt().dropna()["value"]
        })
        violin_data = violin_data.melt(var_name="system", value_name="RMSD")
        violin_data["sp"] = ""
        return violin_data

    data1 = prepare_data(df1_wt, df1_nc)
    data2 = prepare_data(df2_wt, df2_nc)

    # --- фигура ---
    
    fig, axes = plt.subplots(2, 1, figsize=(3, 6), sharey=True)

    palette = ["#88BAF2", "#CD1C18"]

    # --- первый subplot ---
    sns.violinplot(
        data=data1, x="sp", y="RMSD", hue="system",
        split=True, inner=None, gap=0.04,
        palette=palette, ax=axes[0]
    )

    # --- второй subplot ---
    sns.violinplot(
        data=data2, x="sp", y="RMSD", hue="system",
        split=True, inner=None, gap=0.04,
        palette=palette, ax=axes[1]
    )

    # --- оформление ---
    for ax in axes:
        ax.set_ylim(2.5, ylim)
        ax.set_xticks([])
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.tick_params(labelsize=13)

        # убираем лишние рамки
        ax.spines['bottom'].set_visible(False)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        # убираем локальные легенды
        if ax.get_legend():
            ax.get_legend().remove()

    # --- заголовки ---
    axes[0].set_title(title1)
    axes[1].set_title(title2)

    # --- общая ось ---
    fig.supylabel("Расстояние, Å", fontsize=15,x=-0.08)

    blue_handles = tuple(
        mlines.Line2D([], [], color=palette[0], lw=2) for _ in range(2)
    )
    red_handles = tuple(
        mlines.Line2D([], [], color=palette[1], lw=2) for _ in range(2)
    )

    fig.legend(""
    )

    plt.subplots_adjust(hspace=0.3)
    plt.show()

Отдельно для 343K без МГЭТ

In [ ]:
# палитры
nc_cmap = sns.light_palette("#88BAF2", n_colors=3, as_cmap=True)
nc_palette = nc_cmap(np.linspace(0.3, 1, 3))

wt_cmap = sns.light_palette("#CD1C18", n_colors=4, as_cmap=True)
wt_palette = wt_cmap(np.linspace(0.3, 1, 4))

# проверка Frame
if "Frame" not in ser_wt.columns:
    ser_wt.reset_index(names="Frame", inplace=True)
    ser_nc.reset_index(names="Frame", inplace=True)
    asp_wt.reset_index(names="Frame", inplace=True)
    asp_nc.reset_index(names="Frame", inplace=True)

# --- фигура + grid ---
fig = plt.figure(figsize=(8, 6))
gs = fig.add_gridspec(2, 1, hspace=0.3)

# --- broken axes ---
bax1 = brokenaxes(
    ylims=((2, 5.5), (8.2, 9.2)),
    subplot_spec=gs[0]
)

ax2 = fig.add_subplot(gs[1])

# --- plotting ---
plot_replicas(ser_wt, wt_palette, bax1)
plot_replicas(ser_nc, nc_palette, bax1)

plot_replicas(asp_wt, wt_palette, ax2)
plot_replicas(asp_nc, nc_palette, ax2)

# --- линии ---
for bax in [bax1, ax2]:
    bax.axhline(y=3.5, color="grey", linestyle="--", linewidth=0.6)

# --- заголовки ---
bax1.set_title("131Ser Oγ – Nε₂ 209His")
ax2.set_title("209His Nδ₁ – Oδ₁ 177Asp")

# --- ticks ---
for bax in bax1.axs:
    bax.tick_params(labelsize=13)

# --- легенда ---
blue_handles = tuple(
    mlines.Line2D([], [], color=c, lw=2) for c in nc_palette
)

red_handles = tuple(
    mlines.Line2D([], [], color=c, lw=2) for c in wt_palette
)

fig.legend(
    [red_handles, blue_handles],
    ['WT PHL7', 'trPHL7'],
    handler_map={tuple: HandlerTuple(ndivide=None)},
    bbox_to_anchor=(0.9, 0.98)
)

for ax in bax1.axs:
    ax.spines['right'].set_visible(True)

bax1.axs[0].spines['top'].set_visible(True)


# --- подписи ---
# fig.suptitle(title, fontsize=15)
fig.supxlabel("Время, нс", fontsize=15)
fig.supylabel("Расстояние, Å", fontsize=15)

fig.suptitle("343K", fontsize=15)

plt.show()

#### RMSF

In [1]:
def rmsf_plotit_new_broken(df_wt, df_nc, title="", ylim = (6, 10), bigsize=False, filename=None, show=True, cut=False): #это для графиков RMSF

    # --- подготовка данных ---
    if "Residue" not in df_wt.columns:
        df_wt["Avg"] = df_wt.mean(axis=1)
        df_wt["SE"] = df_wt.drop("Avg", axis=1).sem(axis=1)
        df_wt.reset_index(names="Residue", inplace=True)

        df_nc["Avg"] = df_nc.mean(axis=1)
        df_nc["SE"] = df_nc.drop("Avg", axis=1).sem(axis=1)
        df_nc.reset_index(names="Residue", inplace=True)
        df_nc["Residue"] += 24

    # --- размер фигуры ---
    if bigsize:
        fig = plt.figure(figsize=(20, 4), dpi=200)
    else:
        fig = plt.figure(figsize=(8, 4))

    # --- broken axes ---
    bax = brokenaxes(
        ylims=((0, 3.8), ylim),
        hspace=0.05,
        height_ratios=(1, 4)
    )

    # --- линии ---
    bax.plot(df_wt["Residue"], df_wt["Avg"], color="#CD1C18", label="WT PHL7")
    bax.plot(df_nc["Residue"], df_nc["Avg"], color="#88BAF2", label="trPHL7")

    # --- области SE ---
    bax.fill_between(
        df_wt["Residue"],
        df_wt["Avg"] - df_wt["SE"],
        df_wt["Avg"] + df_wt["SE"],
        color="#CD1C18",
        alpha=0.2
    )

    bax.fill_between(
        df_nc["Residue"],
        df_nc["Avg"] - df_nc["SE"],
        df_nc["Avg"] + df_nc["SE"],
        color="#88BAF2",
        alpha=0.2
    )

    # --- оформление ---
    bax.set_xlim(-10, 265)
    bax.axs[-1].set_yticks(np.arange(0, 3.5, 0.5))
    bax.set_xlabel("Номер остатка", fontsize = 15, labelpad = 20)
    bax.set_ylabel("RMSF, Å", fontsize = 15)

    for ax in bax.axs:
        ax.spines['right'].set_visible(True)
        ax.tick_params(axis='x', labelsize=13)
        ax.tick_params(axis='y', labelsize=13)

    bax.axs[0].spines["top"].set_visible(True)
    plt.title(title, fontsize = 15)

    bax.legend()

    bax.axs[1].axvline(x = 131, linewidth = 1,ls='-.',color="#88BAF2")
    bax.axs[1].axvline(x = 177, linewidth = 1,ls='-.',color="#88BAF2")
    bax.axs[1].axvline(x = 209, linewidth = 1,ls='-.',color="#88BAF2")

    # ticks (если bigsize)
    if bigsize:
        for ax in bax.axs:
            ax.set_xticks(np.arange(0, 270, 5))

    # --- show/save ---
    if filename:
        plt.savefig(filename, bbox_inches="tight")

    if show:
        plt.show()

    return None

In [ ]:
rmsf_wt = pd.read_csv(home_path+system_wt_mhet+"/343K/tables/rmsf.tsv", sep = "\t")
rmsf_nc = pd.read_csv(home_path+system_nc_mhet+"/343K/tables/rmsf.tsv", sep = "\t")


rmsf_plotit_new_broken(rmsf_wt, rmsf_nc.drop("Rep4", axis = 1), title = "+МГЭТ 300K", ylim = (6,8))

$\Delta RMSF$

In [ ]:
def delta_rmsf_prep (wt, nc):
    wt_nc = pd.DataFrame({"WT":wt[23:232].mean(axis = 1).reset_index(drop=True), 
                          "NC":nc.mean(axis=1)})
    wt_nc["Delta"] = wt_nc["NC"]-wt_nc["WT"]
    return wt_nc

def delta_rmsf_plotit (wt_nc_df, title = "", filename = ""):
    cmap = mcolors.LinearSegmentedColormap.from_list(
    "my_cmap",
    ["#88BAF2", "#ffffff", "#CD1C18"]
    )

    x = wt_nc_df.index.values+24
    y = wt_nc_df["Delta"].values

    factor = 50
    x_dense = np.linspace(x.min(), x.max(), len(x) * factor)
    y_dense = np.interp(x_dense, x, y)

    # создаём сегменты
    points = np.array([x_dense, y_dense]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)

    y_mid = (y_dense[:-1] + y_dense[1:]) / 2

    # нормализация (чтобы 0 был центром)
    norm = mcolors.TwoSlopeNorm(vmin=y_dense.min(), vcenter=0, vmax=y_dense.max())

    lc = LineCollection(segments, cmap=cmap, norm=norm)
    lc.set_array(y_mid)
    lc.set_linewidth(2)
    lc.set_capstyle('round')
    lc.set_joinstyle('round')

    fig, ax = plt.subplots()
    ax.add_collection(lc)

    ax.set_xlim(20, 235)
    ax.set_ylim(-1.5, 2)

    ax.plot(x, y, color='black', linewidth=2.5)   # толстый контур
    ax.add_collection(lc)   

    plt.axvline(x = 131, linewidth = 1,ls='-.',color="#88BAF2")
    plt.axvline(x = 177, linewidth = 1,ls='-.',color="#88BAF2")
    plt.axvline(x = 209, linewidth = 1,ls='-.',color="#88BAF2")

    plt.title(title, fontsize = 15)
    plt.xlabel("Номер остатка", fontsize = 15)
    plt.ylabel(r"$\Delta RMSF$, Å", fontsize = 15)
    plt.yticks(fontsize = 13)
    plt.xticks(fontsize = 13)
    plt.show()

    return None

In [ ]:
wt_rmsf_300 = pd.read_csv(home_path+"7NEI/300K/tables/rmsf.tsv", sep = "\t")
nc_rmsf_300 = pd.read_csv(home_path+"NC_7NEI/300K/tables/rmsf.tsv", sep = "\t")
wt_rmsf_343 = pd.read_csv(home_path+"7NEI/343K/tables/rmsf.tsv", sep = "\t")
nc_rmsf_343 = pd.read_csv(home_path+"NC_7NEI/343K/tables/rmsf.tsv", sep = "\t")
wt_mhet_rmsf_300 = pd.read_csv(home_path+"7NEI_MHET/300K/tables/rmsf.tsv", sep = "\t")
nc_mhet_rmsf_300 = pd.read_csv(home_path+"NC_7NEI_MHET/300K/tables/rmsf.tsv", sep = "\t")
wt_mhet_rmsf_343 = pd.read_csv(home_path+"7NEI_MHET/343K/tables/rmsf.tsv", sep = "\t")
nc_mhet_rmsf_343 = pd.read_csv(home_path+"NC_7NEI_MHET/343K/tables/rmsf.tsv", sep = "\t")



wt_nc_300 = delta_rmsf_prep(wt_rmsf_300,nc_rmsf_300)
wt_nc_343 = delta_rmsf_prep(wt_rmsf_343,nc_rmsf_343)
wt_nc_mhet_300 = delta_rmsf_prep(wt_mhet_rmsf_300,nc_mhet_rmsf_300)
wt_nc_mhet_343 = delta_rmsf_prep(wt_mhet_rmsf_343,nc_mhet_rmsf_343)

In [ ]:
delta_rmsf_plotit(wt_nc_343, title = "343K")